Подготовка колаба, докачиваем наши скрипты и делаем енв

In [ ]:
!git clone -b checkpoint3 --single-branch https://github.com/Opuntia-cactaceae/mod_translation.git
!mkdir -p translator_code
!cp -r mod_translation/checkpoint3/llm_tranlator/translator_code/* translator_code/

In [4]:
!cp mod_translation/checkpoint3/llm_tranlator/_env .env

api_key = "Ключ от грока"

with open(".env", "a") as f:
    f.write(f"\nGROQ_API_KEY={api_key}\n")

Произведем расчет метрики для бейслайна.
В нашем случае была использована модель llama-3.1-8b-instant с groq.com. У нее большое кол-во токенов и ее потенциально можно запустить локально. Перевод был с en на ru.

Строки уже были переведены заранее, можно свободно запускать расчет метрики.

In [5]:
!pip install groq
!pip install unbabel-comet
import pathlib
from typing import List
from translator_code.db_worker import load_eval_data_for_model
import numpy as np
import pandas as pd
from tqdm import tqdm
from translator_code.calc_metric import (
    download_model,
    load_from_checkpoint,
    ModelCometScorer,
    ScoreSet
)

DB_DIR = pathlib.Path("./db")
DB_FILENAME = "parallel_pairs_stream.sqlite"
DB_PATH = DB_DIR / DB_FILENAME

# Ссылка на БД на Google Drive
GDRIVE_URL = "https://drive.google.com/file/d/1rhrsSFS-4KZbK0HYTkmerjXpEsisf7NC/view?usp=share_link"

TARGET_MODEL_NAME = "llama-3.1-8b-instant"

DB_SOURCE_LANG = "en"   # язык исходного текста в формате для бд
DB_TARGET_LANG = "ru"   # язык таргета

COMET_MODEL_NAME = "Unbabel/wmt22-comet-da"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 28.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
    

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

Скачиваем датасет с гугл диска, если он не был скачан для перевода строк.

In [ ]:
def ensure_gdown_installed():
    """Устанавливает gdown, если его нет, и возвращает модуль."""
    try:
        import gdown
    except ImportError:
        print("gdown не найден")
    finally:
        import gdown
    return gdown


def normalize_gdrive_url(url: str) -> str:
    """
    Принимает ссылку вида
      https://drive.google.com/file/d/<ID>/view?usp=...
    и возвращает
      https://drive.google.com/uc?id=<ID>
    чтобы gdown мог её скачать.
    """
    if "uc?id=" in url:
        return url
    import re
    m = re.search(r"/d/([^/]+)/", url)
    if not m:
        return url
    file_id = m.group(1)
    return f"https://drive.google.com/uc?id={file_id}"


def download_db_if_needed():
    """
    Проверяет наличие БД по пути DB_PATH.
    Если файла нет — создаёт ./db и качает файл с Google Drive.
    """
    if DB_PATH.exists():
        print(f"База уже существует: {DB_PATH}")
        return

    if not GDRIVE_URL:
        raise ValueError("Не задана GDRIVE_URL с ссылкой на БД в Google Drive.")

    DB_DIR.mkdir(parents=True, exist_ok=True)

    gdown = ensure_gdown_installed()
    url = normalize_gdrive_url(GDRIVE_URL)

    print(f"Скачиваю базу из Google Drive в {DB_PATH} ...")
    gdown.download(url, str(DB_PATH), quiet=False)

    if not DB_PATH.exists():
        raise RuntimeError("Не удалось скачать БД с Google Drive.")
    print("База успешно скачана.")

Производим расчет метрики по переведенным строкам.

In [ ]:

def load_comet_scorer(model_name: str) -> ModelCometScorer:
    """Загрузка COMET-модели и обёртки ModelCometScorer."""
    model_path = download_model(model_name)
    comet_model = load_from_checkpoint(model_path)
    return ModelCometScorer(comet_model)


def compute_overall_metric(df: pd.DataFrame, comet_scorer: ModelCometScorer) -> float:
    """
    Считает final_score для всех строк и возвращает одно число — среднее.
    Теперь всё делается через ScoreSet (COMET + теги внутри).
    """
    if df.empty:
        raise ValueError(
            "В датафрейме нет строк для оценки — возможно, нет данных для этой модели."
        )

    refs = df["ref"].tolist()
    cands = df["cand"].tolist()

    if "src" in df.columns:
        srcs = df["src"].tolist()
    else:
        srcs = [None] * len(df)

    score_set = ScoreSet(
        refs=refs,
        cands=cands,
        srcs=srcs,
        comet_scorer=comet_scorer,
        precomputed_text_scores=None,
        comet_batch_size=64,
        show_progress=True,
        parallel=True,
    )

    result = score_set.compute()
    return float(np.mean(result.final_score))


# Проверяем/скачиваем базу
download_db_if_needed()

# Грузим данные для модели
df_eval = load_eval_data_for_model(
    DB_PATH,
    TARGET_MODEL_NAME,
    src_lang=DB_SOURCE_LANG,
    tgt_lang=DB_TARGET_LANG,
)

print(f"Найдено строк для оценки модели '{TARGET_MODEL_NAME}': {len(df_eval)}")

if df_eval.empty:
    raise SystemExit("Нет данных (строк с ref) для заданной модели — оценивать нечего.")

# Загружаем COMET и считаем метрику через новый ScoreSet
comet_scorer = load_comet_scorer(COMET_MODEL_NAME)
overall_final_score = compute_overall_metric(df_eval, comet_scorer)

print(f"\nИтоговый средний final_score для модели '{TARGET_MODEL_NAME}': {overall_final_score:.4f}")

База уже существует: db/parallel_pairs_stream.sqlite
Найдено строк для оценки модели 'llama-3.1-8b-instant': 1000


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
Scoring (COMET + tags): 100%|██████████| 1000/1000 [1:06:02<00:00,  3.96s/it]


Итоговый средний final_score для модели 'llama-3.1-8b-instant': 0.6959


Выполнение в коллабе занимает +-  60 минут. В итоге получили 0.6959. Что можно сказать:
Не плохо, не знаю на сколько хорошо, но не плохо.
Я уже пробовал переводить себе моды с текущим пайплайном (правда с использованием всех доступных моделей, чтобы было быстрее и вкуснее) и играть вполне можно. Кач-во перевода у базовой модельки звезд с неба не хватает, но потенциал для дальнейшего промпт инжиниринга есть.